# Lab 11 · Kỷ luật đo lường cho LLM: schema, nhãn tay & hậu kiểm

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành tuần 11**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo buổi 11 chạy trọn pipeline Gemini trên 24 review. Lab này luyện phần
**khó và quan trọng nhất** của hợp phần LLM — và cố tình không cần API key: bạn sẽ chọn mẫu
thông minh, viết schema, rồi **thẩm định một bộ output LLM mô phỏng có lỗi cài sẵn** bằng
nhãn tay — đúng việc nhóm bạn sắp làm với ≥100 nhãn của bài tập lớn.

*Lab ~50 phút; 30 phút cuối là BTL clinic (mục cuối notebook).*

## Cách làm việc trong buổi lab

- Bài tập được chia bước; mỗi bước có ô `TODO` và phần kiểm tra `assert` — chạy qua hết
  `assert` nghĩa là bạn làm đúng.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — micro-exercise 🔒
  cuối giờ đo đúng các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn 🔓: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Bạn kẹt quá 3 phút ở một bước: gọi trợ giảng.

## Mục tiêu

Sau buổi lab, bạn:

1. Chọn mẫu review "đáng tiền" để gửi LLM và ước lượng chi phí token trước khi gọi.
2. Viết schema Pydantic có enum và thấy nó chặn nhãn bịa **trước khi** vào bảng.
3. Đo accuracy của output LLM trên nhãn tay; phân loại lỗi ngữ nghĩa.
4. Viết một quy tắc hậu kiểm tự động.

In [ ]:
%pip install -q pydantic

## Phần 0 · Chọn mẫu & ước chi phí (~12 phút)

Free tier có hạn — không ai gửi cả 690 nghìn review cho model. Chọn mẫu là bước pipeline
thật sự (lab 7 đã thấy: ~59 nghìn comment ngắn vô nghĩa).

In [ ]:
import pandas as pd

URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/data/reviews.csv.gz")
rv = pd.read_csv(URL, usecols=["id", "comments"]).dropna(subset=["comments"])
rv["comments"] = rv["comments"].str.replace("<br/>", " ", regex=False)

# TODO: lọc comment dài >= 50 ký tự, rồi lấy mẫu 100 review với random_state=42
du_dai = ...
mau = ...

# --- Ô kiểm tra ---
assert len(mau) == 100
assert mau["comments"].str.len().min() >= 50
print(f"Chọn 100/{len(du_dai):,} review đủ dài — mẫu tái lập được nhờ random_state.")

In [ ]:
# Ước lượng token: quy tắc ngón tay cái ~4 ký tự/token (đủ cho ước lượng chi phí)
# TODO: tính tổng token ước lượng của 100 comment trong mẫu (cộng len // 4)
tong_token = ...

# Giá flash-lite: $0.25 / 1M token vào. TODO: ước chi phí phần dữ liệu (USD)
chi_phi = ...

# --- Ô kiểm tra ---
assert tong_token == int((mau["comments"].str.len() // 4).sum())
assert chi_phi < 0.01
print(f"~{tong_token:,} token dữ liệu ≈ ${chi_phi:.5f} — chưa tính prompt; mẫu 100 gần như miễn phí.")

## Phần 1 · Schema là hàng rào đầu tiên (~12 phút)

In [ ]:
from pydantic import BaseModel, ValidationError
from typing import Literal

Aspect = Literal["location", "cleanliness", "host", "noise", "amenities", "value"]

# TODO: viết class ReviewInfo(BaseModel) gồm 4 trường:
#   sentiment: Literal 3 giá trị "positive" / "mixed" / "negative"
#   aspects_positive: list[Aspect]
#   aspects_negative: list[Aspect]
#   language: str
class ReviewInfo(BaseModel):
    ...

# --- Ô kiểm tra ---
ok = ReviewInfo.model_validate_json(
    '{"sentiment": "mixed", "aspects_positive": ["location"], '
    '"aspects_negative": ["noise"], "language": "en"}')
assert ok.sentiment == "mixed" and ok.aspects_negative == ["noise"]
try:
    ReviewInfo.model_validate_json(
        '{"sentiment": "happy", "aspects_positive": [], "aspects_negative": [], "language": "en"}')
    raise SystemError("Lẽ ra phải bị chặn!")
except ValidationError:
    print("Schema chặn đúng nhãn 'happy' ngoài enum — hàng rào hoạt động.")

## Phần 2 · Thẩm định output LLM bằng nhãn tay (~26 phút)

Dưới đây là **10 review thật** của Santiago, nhãn tay (GOLD) do "nhóm" gán, và **10 output
mô phỏng kiểu LLM** — trong đó cài sẵn lỗi, như dữ liệu thật sẽ có. Việc của bạn: để schema
và nhãn tay tự bắt chúng.

In [ ]:
REVIEWS = {
 1: "Place is great for a family, all clean, good location, bit noisy as the main avenue is behind, but we had a great time.",
 2: "Apartment was irrelevant with pictures and not clean too so we cancelled our reservation. Alvaro helped us about cancellation process.",
 3: "Cristian fue muy amable en todo momento, el lugar como se describia. La zona con muy buena movilidad. Muy recomendable. Gracias Cristian",
 4: "Hermoso alojamiento! Lo pasamos re bien mi hija y yo. Es un poco ruidosa la zona si se abre la ventana. La vista es bella y esta muy bien ubicado. Sin duda volveriamos",
 5: "Great location but apartment needs some attention to detail. Cable TV and wifi was out of service. Communication with host was poor. Bedding was not optimal.",
 6: "Good WiFi, great location and centrally located. Shower was hot and had good pressure and there was enough space for two people for 6 days.",
 7: "Es tal cual las fotos, buena ubicacion, tranquilo y sin ruido. Es en un piso 15 por si le temen a las alturas.",
 8: ".",
 9: "Nice place in cool region. Very noisy environment and apartment is not very clean.",
 10: "location was great, wifi was spotty. But overall not a bad place.",
}

# Nhãn tay của "nhóm" (GOLD) — quy ước: review rỗng/vô nghĩa -> mixed, không khía cạnh, "unknown"
GOLD = {
 1: {"sentiment": "mixed",    "language": "en"},
 2: {"sentiment": "negative", "language": "en"},
 3: {"sentiment": "positive", "language": "es"},
 4: {"sentiment": "mixed",    "language": "es"},
 5: {"sentiment": "negative", "language": "en"},
 6: {"sentiment": "positive", "language": "en"},
 7: {"sentiment": "positive", "language": "es"},
 8: {"sentiment": "mixed",    "language": "unknown"},
 9: {"sentiment": "mixed",    "language": "en"},
 10: {"sentiment": "mixed",   "language": "en"},
}

# Output mô phỏng kiểu LLM (JSON thô) — CÓ LỖI CÀI SẴN, đừng sửa
LLM_OUT = {
 1: '{"sentiment": "mixed", "aspects_positive": ["cleanliness", "location"], "aspects_negative": ["noise"], "language": "en"}',
 2: '{"sentiment": "negative", "aspects_positive": ["host"], "aspects_negative": ["photos", "cleanliness"], "language": "en"}',
 3: '{"sentiment": "positive", "aspects_positive": ["host", "location"], "aspects_negative": [], "language": "es"}',
 4: '{"sentiment": "positive", "aspects_positive": ["location"], "aspects_negative": [], "language": "es"}',
 5: '{"sentiment": "negative", "aspects_positive": ["location"], "aspects_negative": ["amenities", "host"]}',
 6: '{"sentiment": "positive", "aspects_positive": ["amenities", "location"], "aspects_negative": [], "language": "en"}',
 7: '{"sentiment": "positive", "aspects_positive": ["location"], "aspects_negative": [], "language": "es"}',
 8: '{"sentiment": "mixed", "aspects_positive": [], "aspects_negative": [], "language": "unknown"}',
 9: '{"sentiment": "negative", "aspects_positive": ["location"], "aspects_negative": ["noise", "cleanliness"], "language": "en"}',
 10: '{"sentiment": "negative", "aspects_positive": ["location"], "aspects_negative": ["amenities"], "language": "en"}',
}
print(len(REVIEWS), "review · 10 output mô phỏng · 10 nhãn tay")

### Bước 1 · Validate hàng loạt — schema bắt được gì?

In [ ]:
# TODO: duyệt LLM_OUT, validate từng JSON bằng ReviewInfo;
#       hợp lệ -> bỏ vào dict hop_le[id], lỗi -> bỏ id vào list loi_schema
hop_le = {}
loi_schema = []
for rid, raw in LLM_OUT.items():
    ...

# --- Ô kiểm tra ---
assert len(hop_le) == 8 and sorted(loi_schema) == [2, 5]
print(f"Schema chặn {len(loi_schema)} output: id {loi_schema} — xem lỗi của chúng ở ô sau.")

In [ ]:
# Soi lý do từng output bị chặn (chạy và đọc)
for rid in loi_schema:
    try:
        ReviewInfo.model_validate_json(LLM_OUT[rid])
    except ValidationError as e:
        print(f"--- id {rid}: {e.errors()[0]['type']} tại {e.errors()[0]['loc']}")

Id 2 bịa nhãn `photos` ngoài enum; id 5 **thiếu trường** `language`. Cả hai bị chặn
*trước khi* lọt vào bảng — rẻ hơn nhiều so với dọn rác sau. Trong pipeline thật, các output
bị chặn sẽ được **gọi lại** (retry) hoặc ghi vào danh sách lỗi.

### Bước 2 · Accuracy sentiment trên nhãn tay

In [ ]:
# TODO: trên 8 output hợp lệ, đếm số id có sentiment TRÙNG với GOLD; tính accuracy
so_dung = ...
acc = ...

# --- Ô kiểm tra ---
assert so_dung == 5 and acc == 0.625
sai = [rid for rid in hop_le if hop_le[rid].sentiment != GOLD[rid]["sentiment"]]
print(f"Accuracy sentiment: {acc:.0%} — sai ở id {sorted(sai)}.")

### Bước 3 · Nhìn vào chỗ sai — phân loại lỗi

Đọc lại 3 review bị gán sai (chạy ô dưới) và đối chiếu bảng phân loại:

| id | GOLD | LLM | Kiểu lỗi |
|---|---|---|---|
| 4 | mixed | positive | **bỏ sót caveat** — lời khen dài che câu chê tiếng ồn |
| 9 | mixed | negative | **ca biên giới** — nhóm gán mixed ("nice place" + 2 chê); tranh luận được |
| 10 | mixed | negative | **bẫy phủ định** — "not a *bad* place" là khen nhẹ, model đọc thành chê |

Ca số 9 quan trọng: có những review mà *người với người* cũng gán khác nhau. Vì thế
hướng dẫn gán nhãn của nhóm (đã thống nhất trước) mới là "chân lý" — và phải **nhất quán**.

In [ ]:
for rid in [4, 9, 10]:
    print(f"[{rid}] GOLD={GOLD[rid]['sentiment']:8s} LLM={hop_le[rid].sentiment:8s} | {REVIEWS[rid][:90]}")

### Bước 4 · Hậu kiểm tự động

In [ ]:
# Quy tắc hậu kiểm: output nào tự mâu thuẫn — sentiment "positive" mà lại có
# aspects_negative không rỗng (hoặc "negative" mà có aspects_positive không rỗng)?
# TODO: đếm số output hợp lệ vi phạm quy tắc trên
mau_thuan = ...

# --- Ô kiểm tra ---
assert sorted(mau_thuan) == [9, 10]
print(f"Hậu kiểm bắt id {sorted(mau_thuan)} — trùng với 2/3 ca gán sai ở Bước 3!")

Một quy tắc hậu kiểm 3 dòng — không cần nhãn tay — đã khoanh được 2 trong 3 ca sai
(để người đọc lại). Nhãn tay đo *chất lượng tổng thể*; hậu kiểm chạy *trên từng output mới*
khi pipeline vận hành. Hợp phần LLM của bài tập lớn cần **cả hai**.

## Bài tự làm 🔓 · Chạy thật với API key của bạn

Nếu bạn đã có key (clinic tuần trước): dùng đúng code mẫu của notebook demo buổi 11
(mục 4 — structured output với `ReviewInfo.model_json_schema()`), gọi Gemini cho
**10 review trong `REVIEWS`** rồi: (1) validate như Bước 1; (2) đo accuracy với `GOLD`
như Bước 2; (3) so kết quả model thật với bộ mô phỏng ở lab. Nhớ `time.sleep(4)` giữa
các lời gọi và lưu cache JSON. Ghi lại: model thật đúng/sai ở những id nào — có sập
đúng "bẫy phủ định" số 10 không?

In [ ]:
# Viết bài tự làm của bạn ở đây (cần GEMINI_API_KEY trong Colab Secrets)

---

## 🧭 BTL clinic tuần 11 (~30 phút — theo nhóm, trợ giảng đi từng bàn)

Trọng tâm tuần này: **khởi động hợp phần LLM cho đúng ngay từ đầu.**

1. ☐ API key Gemini của người phụ trách nằm trong **Colab Secrets / biến môi trường** —
   tuyệt đối chưa từng xuất hiện trong code hay commit nào (soát bằng `git log -p | grep`).
2. ☐ **Mẫu review** để gán nhãn đã chốt: cách lọc (độ dài, thời gian) + `random_state`,
   đủ ≥100, cân nhắc tỷ lệ ngôn ngữ.
3. ☐ **Schema + enum khía cạnh** bản nháp đã viết (theo lab này); danh mục khía cạnh
   hợp với câu hỏi phân tích của nhóm.
4. ☐ **Hướng dẫn gán nhãn** 5–10 dòng đã viết chung (quy ước ca rỗng, ca mixed, ca đa
   ngôn ngữ); kế hoạch: mỗi review 2 người gán độc lập → bất đồng thì thảo luận.
5. ☐ Baseline không-LLM (từ khoá buổi 7) chạy được trên đúng mẫu đó — sẵn sàng để so.

> Nhóm xong sớm: chạy thử 5 lời gọi Gemini thật với schema của nhóm, xem output sống
> có đúng form không.

## Tóm tắt buổi lab

| Bạn đã làm | Dùng cho |
|---|---|
| Chọn mẫu + ước token trước khi gọi | kế hoạch chi phí hợp phần LLM |
| Schema/enum chặn nhãn bịa, trường thiếu | pipeline LLM của nhóm |
| Accuracy trên nhãn tay + phân loại lỗi | mục "đo chất lượng trên ≥100 nhãn" của đề |
| Hậu kiểm tự mâu thuẫn | vận hành pipeline sau khi nộp |

Buổi lý thuyết tới: **trực quan hoá cơ bản** — các con số này bắt đầu thành hình.